# 30-Day Readmission Risk Prediction

End-to-end ML use case: binary classification to predict 30-day readmission from encounter and patient features.
Uses the healthcare mock dataset (patients, encounters, diagnoses, labs, readmissions). Feature code is shared with `src/python/readmission_model.py` via `readmission_features.py`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

REPO_ROOT = Path.cwd().parent.parent if Path.cwd().name == "python" else Path.cwd().parent
SRC_PYTHON = REPO_ROOT / "src" / "python"
if str(SRC_PYTHON) not in sys.path:
    sys.path.insert(0, str(SRC_PYTHON))
from readmission_features import load_and_feature

DATA_DIR = REPO_ROOT / "data" / "raw"

## Load features

Target: 30-day readmission. Features: LOS, age, diagnosis count, lab summaries, encounter-type dummies (see `src/python/readmission_features.py`).

In [ ]:
X, y, feature_cols = load_and_feature(DATA_DIR)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

## Evaluation

Readmission is usually **imbalanced**: ROC-AUC can look strong while precision at default threshold is poor. **Average precision** (area under the PR curve) is often more informative than ROC alone for rare events.

In [ ]:
print(
    classification_report(
        y_test, y_pred, target_names=["No readmit", "Readmit 30d"], zero_division=0
    )
)
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("Average precision (PR-AUC):", round(average_precision_score(y_test, y_prob), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax[0], name="RandomForest")
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=ax[1], name="RandomForest")
ax[0].set_title("ROC curve")
ax[1].set_title("Precision–recall")
plt.tight_layout()
plt.show()

In [ ]:
imp = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
imp.plot(kind="barh", title="Feature importance (readmission model)")
plt.tight_layout()
plt.show()

## Hyperparameter tuning

GridSearchCV over a small parameter grid (n_estimators, max_depth); scoring can be `roc_auc` or `average_precision` depending on whether you care more about ranking or the positive class under imbalance.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {"n_estimators": [50, 100, 150], "max_depth": [6, 8, 10]}
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 4))
y_prob_tuned = grid.predict_proba(X_test)[:, 1]
print("Test ROC-AUC (tuned):", round(roc_auc_score(y_test, y_prob_tuned), 4))
print(
    "Test average precision (tuned):",
    round(average_precision_score(y_test, y_prob_tuned), 4),
)

## Summary

Interpretable baseline for 30-day readmission. **Next steps:** threshold tuning for precision/recall tradeoffs, `GridSearchCV(scoring="average_precision")` under imbalance, richer features (comorbidity indices, prior utilization, facility), and calibration if probabilities drive interventions.